In [ ]:

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torchvision.transforms import v2
from torch.utils.data import DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from medmnist import DermaMNIST, INFO

# CONFIG
LR           = 0.001
EPOCHS       = 50
BATCH_SIZE   = 128
MILESTONES   = [20, 35, 45]
GAMMA        = 0.5
NUM_MC       = 100
WEIGHT_DECAY = 1e-4
PATIENCE     = 5
RESULTS_FILE = "derma_results.json"
CKPT_DIR     = "derma_checkpoints"
FIG_DIR      = "derma_figures"

BNN_PARAMS = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": False,
    "moped_delta": 0.5,
}

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

device = torch.device(
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available() else "cpu"
)
print(f"Device : {device}")

#Data

transform_aug = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation(15),
    v2.ColorJitter(brightness=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

transform_eval = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

info        = INFO["dermamnist"]
class_names = list(info["label"].values())
N_CLASSES   = len(class_names)

trainset_noaug = DermaMNIST(split="train", download=True, size=28, transform=transform_eval)
trainset_aug   = DermaMNIST(split="train", download=True, size=28, transform=transform_aug)
valset         = DermaMNIST(split="val",   download=True, size=28, transform=transform_eval)
testset        = DermaMNIST(split="test",  download=True, size=28, transform=transform_eval)

valloader  = DataLoader(valset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

N_TRAIN = len(trainset_noaug)
print(f"Trainset : {N_TRAIN} | Val : {len(valset)} | Test : {len(testset)}")
print(f"Classes  : {class_names}")

#Balancing training set
BALANCE_SEED          = 42
MAX_OVERSAMPLE_FACTOR = 10

def print_class_distribution(labels, title):
    labels = np.asarray(labels).flatten()
    print(f"\n{title}")
    for c, name in enumerate(class_names):
        n = int((labels == c).sum())
        print(f"  [{c}] {name:<48s} {n:>6d}")
    print(f"  {'TOTAL':<52s} {len(labels):>6d}")


def build_balanced_indices(labels, target_per_class=None, max_oversample_factor=MAX_OVERSAMPLE_FACTOR, seed=BALANCE_SEED):
    labels = np.asarray(labels).flatten()
    rng    = np.random.default_rng(seed)

    counts = np.array([int((labels == c).sum()) for c in range(N_CLASSES)])
    if target_per_class is None:
        target_per_class = int(np.median(counts))

    indices = []
    for c in range(N_CLASSES):
        idx_c = np.where(labels == c)[0]
        n_c   = len(idx_c)

        if n_c >= target_per_class:
            chosen = rng.choice(idx_c, size=target_per_class, replace=False)
        else:
            n_final = min(target_per_class, n_c * max_oversample_factor)
            chosen  = rng.choice(idx_c, size=n_final, replace=True)

        indices.append(chosen)

    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return indices.tolist(), target_per_class


train_labels_raw = trainset_noaug.labels.flatten()
print_class_distribution(train_labels_raw, "Distribution originale du trainset :")

balanced_idx, TARGET_PER_CLASS = build_balanced_indices(train_labels_raw)
print(f"\ntarget per class: {TARGET_PER_CLASS}")
print(f"max oversample factor: x{MAX_OVERSAMPLE_FACTOR}")

trainset_noaug = torch.utils.data.Subset(trainset_noaug, balanced_idx)
trainset_aug   = torch.utils.data.Subset(trainset_aug,   balanced_idx)

train_labels_balanced = train_labels_raw[balanced_idx]
print_class_distribution(train_labels_balanced, "Distribution du trainset apres equilibrage :")

N_TRAIN = len(trainset_noaug)
print(f"\nTrainset (equilibre): {N_TRAIN} | Val : {len(valset)} | Test : {len(testset)}")

print_class_distribution(valset.labels.flatten(),  "Distribution du valset :")
print_class_distribution(testset.labels.flatten(), "Distribution du testset :")


# Models

def build_bnn_effnet():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, N_CLASSES)
    dnn_to_bnn(net, BNN_PARAMS)
    return net.to(device)

def build_bnn_mobilenet():
    net = torchvision.models.mobilenet_v3_small(progress=False)
    net.classifier[3] = nn.Linear(1024, N_CLASSES)
    dnn_to_bnn(net, BNN_PARAMS)
    return net.to(device)

# TRAINING

def train_bnn(trainset, build_fn, tag="", weight_decay=0.0, early_stopping=False):
    net = build_fn()
    loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
    n_train = len(trainset)
    train_losses, val_losses = [], []

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    stopped_epoch  = EPOCHS

    for epoch in range(EPOCHS):
        net.train()
        run_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
            optimizer.zero_grad()
            out = net(inputs)
            kl  = get_kl_loss(net)
            loss = criterion(out, labels) + kl / n_train
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
        train_losses.append(run_loss / len(loader))
        scheduler.step()

        net.eval()
        run_val = 0.0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
                run_val += criterion(net(inputs), labels).item()
        val_loss = run_val / len(valloader)
        val_losses.append(val_loss)

        if (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:
            print(f"  [{tag}] epoch {epoch+1}/{EPOCHS} — train {train_losses[-1]:.3f} — val {val_loss:.3f}")

        if early_stopping:
            if val_loss < best_val_loss:
                best_val_loss  = val_loss
                best_state     = {k: v.cpu().clone() for k, v in net.state_dict().items()}
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= PATIENCE:
                    stopped_epoch = epoch + 1
                    print(f"  [{tag}] Early stopping à epoch {stopped_epoch} (patience={PATIENCE})")
                    net.load_state_dict({k: v.to(device) for k, v in best_state.items()})
                    break

    return net, train_losses, val_losses, stopped_epoch

# UNCERTAINTY / ENTROPY

def compute_mc_outputs(net, dataset):
    loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
    net.eval()
    mc_runs = []
    with torch.no_grad():
        for _ in range(NUM_MC):
            batch_probs = []
            for inputs, _ in loader:
                inputs = inputs.to(device)
                batch_probs.append(F.softmax(net(inputs), dim=-1).cpu().numpy())
            mc_runs.append(np.concatenate(batch_probs, axis=0))
    return np.stack(mc_runs, axis=0)  # (MC, N, C)


def entropy_scores(mc_outputs):
    mean_p = mc_outputs.mean(axis=0)
    H      = -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1)
    exp_H  = -np.mean(np.sum(mc_outputs * np.log(mc_outputs + 1e-10), axis=-1), axis=0)
    MI     = H - exp_H
    aleat  = H - MI
    return H, MI, aleat


def entropy_per_class(mc_outputs, labels):
    H, MI, aleat = entropy_scores(mc_outputs)
    h_c, mi_c, al_c = [], [], []
    for c in range(N_CLASSES):
        mask = labels == c
        h_c.append(float(H[mask].mean())     if mask.any() else 0.0)
        mi_c.append(float(MI[mask].mean())   if mask.any() else 0.0)
        al_c.append(float(aleat[mask].mean()) if mask.any() else 0.0)
    return {"H": h_c, "MI": mi_c, "H_MI": al_c}


# EVALUATION

def evaluate_loader(net, loader):
    net.eval()
    all_preds, all_labels, all_outputs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.squeeze(1).cpu().numpy()
            mc = [F.softmax(net(inputs), dim=-1).cpu().numpy() for _ in range(NUM_MC)]
            mc_out  = np.stack(mc, axis=0)
            mean_p  = mc_out.mean(axis=0)
            all_preds.append(mean_p.argmax(axis=-1))
            all_labels.append(labels)
            all_outputs.append(mc_out)
    preds   = np.concatenate(all_preds,  axis=0)
    labels  = np.concatenate(all_labels, axis=0)
    outputs = np.concatenate(all_outputs, axis=1)
    acc     = float((preds == labels).mean())
    return preds, labels, outputs, acc


def compute_auce(mc_outputs, labels, n_bins=20):
    H, _, _  = entropy_scores(mc_outputs)
    preds    = mc_outputs.mean(axis=0).argmax(axis=-1)
    correct  = (preds == labels).astype(float)
    order    = np.argsort(H)
    H_sorted = H[order]
    cor_sort = correct[order]
    N = len(H_sorted)
    bin_size = N // n_bins
    errors, thresholds = [], []
    for i in range(n_bins):
        thresh = H_sorted[min((i + 1) * bin_size - 1, N - 1)]
        err = 1.0 - cor_sort[:min((i + 1) * bin_size, N)].mean()
        errors.append(float(err))
        thresholds.append(float(thresh / (H_sorted.max() + 1e-10)))
    auce = float(np.trapezoid(errors, thresholds))
    return auce, thresholds, errors


def compute_ace(mc_outputs, labels, n_bins=20):
    mean_p  = mc_outputs.mean(axis=0)
    conf    = mean_p.max(axis=-1)
    correct = (mean_p.argmax(axis=-1) == labels).astype(float)
    order   = np.argsort(conf)
    conf_s  = conf[order]
    cor_s   = correct[order]
    N = len(conf_s)
    bin_size = max(N // n_bins, 1)
    accs, confs = [], []
    for i in range(n_bins):
        sl = slice(i * bin_size, min((i + 1) * bin_size, N))
        if cor_s[sl].size == 0:
            continue
        accs.append(float(cor_s[sl].mean()))
        confs.append(float(conf_s[sl].mean()))
    ace = float(np.mean(np.abs(np.array(accs) - np.array(confs))))
    return ace, confs, accs


def compute_uncertain_when_inaccurate(mc_outputs, labels, n_thresholds=50):
    H, _, _    = entropy_scores(mc_outputs)
    preds      = mc_outputs.mean(axis=0).argmax(axis=-1)
    inaccurate = (preds != labels)
    thresholds = np.linspace(H.min(), H.max(), n_thresholds)
    fracs, vals = [], []
    for t in thresholds:
        uncertain = H >= t
        fracs.append(float(uncertain.mean()))
        vals.append(float((uncertain & inaccurate).sum() / max(inaccurate.sum(), 1)))
    return fracs, vals


def compute_conf_vs_acc(mc_outputs, labels, n_thresholds=50):
    mean_p  = mc_outputs.mean(axis=0)
    conf    = mean_p.max(axis=-1)
    correct = (mean_p.argmax(axis=-1) == labels).astype(float)
    thresholds = np.linspace(conf.min(), conf.max(), n_thresholds)
    threshs, frac_acc = [], []
    for t in thresholds:
        mask = conf >= t
        if mask.sum() == 0:
            continue
        threshs.append(float(t))
        frac_acc.append(float(correct[mask].mean()))
    return threshs, frac_acc


def accuracy_per_class(preds, labels):
    accs = []
    for c in range(N_CLASSES):
        mask = labels == c
        accs.append(float((preds[mask] == c).mean()) if mask.any() else 0.0)
    return accs

#Configs
MODEL_CONFIGS = {
    "bnn_weight_decay": {
        "build_fn":       build_bnn_effnet,
        "trainset":       trainset_noaug,
        "weight_decay":   WEIGHT_DECAY,
        "early_stopping": False,
    },
    "bnn_aug": {
        "build_fn":       build_bnn_effnet,
        "trainset":       trainset_aug,
        "weight_decay":   0.0,
        "early_stopping": False,
    },
    "bnn_small": {
        "build_fn":       build_bnn_mobilenet,
        "trainset":       trainset_noaug,
        "weight_decay":   0.0,
        "early_stopping": False,
    },
    "bnn_early_stopping": {
        "build_fn":       build_bnn_effnet,
        "trainset":       trainset_noaug,
        "weight_decay":   0.0,
        "early_stopping": True,
    },
}

#main loop
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f"Results loaded from {RESULTS_FILE}")
else:
    all_results = {}


def save_results():
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)


for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'='*70}\n  MODEL : {model_name}\n{'='*70}")

    if model_name in all_results:
        print(f"  Already calculated, skip.")
        continue

    net, train_losses, val_losses, stopped_epoch = train_bnn(
        trainset       = cfg["trainset"],
        build_fn       = cfg["build_fn"],
        tag            = model_name,
        weight_decay   = cfg["weight_decay"],
        early_stopping = cfg["early_stopping"],
    )

    ckpt_path = os.path.join(CKPT_DIR, f"{model_name}.pth")
    torch.save({
        "model_state_dict": net.state_dict(),
        "train_losses":     train_losses,
        "val_losses":       val_losses,
        "stopped_epoch":    stopped_epoch,
    }, ckpt_path)

    val_preds,  val_labels,  val_mc,  val_acc  = evaluate_loader(net, valloader)
    test_preds, test_labels, test_mc, test_acc = evaluate_loader(net, testloader)

    H_val, MI_val, aleat_val = entropy_scores(val_mc)
    ent_by_class_val         = entropy_per_class(val_mc, val_labels)

    auce_val, auce_thresh, auce_err = compute_auce(val_mc, val_labels)
    ace_val, ace_confs, ace_accs    = compute_ace(val_mc, val_labels)
    ui_fracs, ui_vals               = compute_uncertain_when_inaccurate(val_mc, val_labels)
    ca_threshs, ca_accs             = compute_conf_vs_acc(val_mc, val_labels)

    acc_per_class = accuracy_per_class(test_preds, test_labels)

    all_results[model_name] = {
        "stopped_epoch":       stopped_epoch,
        "val_acc":             val_acc,
        "test_acc":            test_acc,
        "val_H_mean":          float(H_val.mean()),
        "val_MI_mean":         float(MI_val.mean()),
        "val_aleatoric_mean":  float(aleat_val.mean()),
        "entropy_by_class_val": ent_by_class_val,
        "acc_per_class_test":  acc_per_class,
        "calibration": {
            "auce":                             auce_val,
            "auce_thresholds":                  auce_thresh,
            "auce_errors":                      auce_err,
            "ace":                              ace_val,
            "ace_confidences":                  ace_confs,
            "ace_accuracies":                   ace_accs,
            "uncertain_when_inaccurate_fracs":  ui_fracs,
            "uncertain_when_inaccurate_vals":   ui_vals,
            "conf_vs_acc_thresholds":           ca_threshs,
            "conf_vs_acc_accs":                 ca_accs,
        },
        "train_losses": train_losses,
        "val_losses":   val_losses,
    }
    save_results()
    print(f"  val_acc={val_acc:.3f} | test_acc={test_acc:.3f} | AUCE={auce_val:.3f} | ACE={ace_val:.3f}")

print(f"\nAll results saved to {RESULTS_FILE}")

MODEL_COLORS = {
    "bnn_weight_decay":   "#4C72B0",
    "bnn_aug":            "#DD8452",
    "bnn_small":          "#55A868",
    "bnn_early_stopping": "#C44E52",
}


def plot_entropy_per_class(model_name, data):
    x     = np.arange(N_CLASSES)
    bar_w = 0.35
    color = MODEL_COLORS[model_name]
    d     = data["entropy_by_class_val"]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - bar_w / 2, d["H"],  bar_w, color=color, label="H",  alpha=0.9)
    ax.bar(x + bar_w / 2, d["MI"], bar_w, color=color, label="MI", hatch="//", alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Nats")
    ax.set_title(f"Entropy / Mutual Information by Class — {model_name} (val)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_entropy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_calibration(model_name, data):
    cal   = data["calibration"]
    color = MODEL_COLORS[model_name]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle(f"Calibration — {model_name}", fontsize=13)
    ax_auce, ax_ace, ax_ui, ax_ca = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

    ax_auce.plot(cal["auce_thresholds"], cal["auce_errors"], color=color,
                 label=f"AUCE={cal['auce']:.1%}")
    ax_auce.set_xlabel("Normalized predictive uncertainty")
    ax_auce.set_ylabel("Error rate")
    ax_auce.set_title("AUCE")
    ax_auce.legend()
    ax_auce.grid(alpha=0.3)

    ax_ace.plot(cal["ace_confidences"], cal["ace_accuracies"], color=color,
                label=f"ACE={cal['ace']:.1%}")
    ax_ace.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Parfait")
    ax_ace.set_xlabel("Confidence")
    ax_ace.set_ylabel("Accuracy")
    ax_ace.set_title("ACE")
    ax_ace.legend()
    ax_ace.grid(alpha=0.3)

    ax_ui.plot(cal["uncertain_when_inaccurate_fracs"],
               cal["uncertain_when_inaccurate_vals"], color=color)
    ax_ui.set_xlabel("Fraction retained (most uncertain)")
    ax_ui.set_ylabel("P(uncertain | inaccurate)")
    ax_ui.set_title("Uncertain when Inaccurate")
    ax_ui.grid(alpha=0.3)

    ax_ca.plot(cal["conf_vs_acc_thresholds"], cal["conf_vs_acc_accs"], color=color)
    ax_ca.set_xlabel("Confidence threshold")
    ax_ca.set_ylabel("Fraction accurate predictions")
    ax_ca.set_title("Confidence vs Accuracy")
    ax_ca.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_calibration.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_accuracy_per_class(model_name, data):
    accs  = data["acc_per_class_test"]
    color = MODEL_COLORS[model_name]
    x     = np.arange(N_CLASSES)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x, [a * 100 for a in accs], color=color, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Accuracy by Class — {model_name} (test)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_accuracy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_comparison_all_models(all_results):
    x     = np.arange(N_CLASSES)
    names = list(all_results.keys())
    n     = len(names)
    bar_w = 0.8 / n

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, name in enumerate(names):
        accs   = all_results[name]["acc_per_class_test"]
        offset = i * bar_w - 0.4 + bar_w / 2
        ax.bar(x + offset, [a * 100 for a in accs], bar_w,
               color=MODEL_COLORS[name], label=name, alpha=0.9)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy by Class — comparison of models (test)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "comparison_accuracy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")



def plot_comparison_calibration(all_results):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Calibration - comparison of models (val)", fontsize=13)
    ax_auce, ax_ace, ax_ui, ax_ca = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

    for name, data in all_results.items():
        cal   = data["calibration"]
        color = MODEL_COLORS[name]
        ax_auce.plot(cal["auce_thresholds"], cal["auce_errors"], color=color,
                     label=f"{name} (AUCE={cal['auce']:.1%})")
        ax_ace.plot(cal["ace_confidences"], cal["ace_accuracies"], color=color,
                    label=f"{name} (ACE={cal['ace']:.1%})")
        ax_ui.plot(cal["uncertain_when_inaccurate_fracs"],
                   cal["uncertain_when_inaccurate_vals"], color=color, label=name)
        ax_ca.plot(cal["conf_vs_acc_thresholds"],
                   cal["conf_vs_acc_accs"], color=color, label=name)

    ax_ace.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Parfait")

    ax_auce.set_xlabel("Normalized predictive uncertainty")
    ax_auce.set_ylabel("Error rate")
    ax_auce.set_title("AUCE")
    ax_auce.legend(fontsize=8)
    ax_auce.grid(alpha=0.3)

    ax_ace.set_xlabel("Confidence")
    ax_ace.set_ylabel("Accuracy")
    ax_ace.set_title("ACE")
    ax_ace.legend(fontsize=8)
    ax_ace.grid(alpha=0.3)

    ax_ui.set_xlabel("Fraction retained (most uncertain)")
    ax_ui.set_ylabel("P(uncertain | inaccurate)")
    ax_ui.set_title("Uncertain when Inaccurate")
    ax_ui.legend(fontsize=8)
    ax_ui.grid(alpha=0.3)

    ax_ca.set_xlabel("Confidence threshold")
    ax_ca.set_ylabel("Fraction accurate predictions")
    ax_ca.set_title("Confidence vs Accuracy")
    ax_ca.legend(fontsize=8)
    ax_ca.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(FIG_DIR, "comparison_calibration.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_comparison_entropy_per_class(all_results):
    x     = np.arange(N_CLASSES)
    names = list(all_results.keys())
    n     = len(names)
    bar_w = 0.8 / (n * 2)

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, name in enumerate(names):
        d         = all_results[name]["entropy_by_class_val"]
        color     = MODEL_COLORS[name]
        offset_H  = (2 * i)     * bar_w - 0.4 + bar_w / 2
        offset_MI = (2 * i + 1) * bar_w - 0.4 + bar_w / 2
        ax.bar(x + offset_H,  d["H"],  bar_w, color=color, label=f"{name} - H",  alpha=0.9)
        ax.bar(x + offset_MI, d["MI"], bar_w, color=color, label=f"{name} - MI", hatch="//", alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Nats")
    ax.set_title("Entropy / Mutual Information by Class - Comparison of Models (val)")
    ax.legend(fontsize=7, ncol=4)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "comparison_entropy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


print("\nGeneration of graphs...")
for model_name, data in all_results.items():
    print(f"\n  {model_name}")
    plot_entropy_per_class(model_name, data)
    plot_calibration(model_name, data)
    plot_accuracy_per_class(model_name, data)

print("\n  Comparison graphs...")
plot_comparison_all_models(all_results)
plot_comparison_calibration(all_results)
plot_comparison_entropy_per_class(all_results)

print("\nDone.")

Device : cuda
Trainset : 7007 | Val : 1003 | Test : 2005
Classes  : ['actinic keratoses and intraepithelial carcinoma', 'basal cell carcinoma', 'benign keratosis-like lesions', 'dermatofibroma', 'melanoma', 'melanocytic nevi', 'vascular lesions']

Distribution originale du trainset :
  [0] actinic keratoses and intraepithelial carcinoma     228
  [1] basal cell carcinoma                                359
  [2] benign keratosis-like lesions                       769
  [3] dermatofibroma                                       80
  [4] melanoma                                            779
  [5] melanocytic nevi                                   4693
  [6] vascular lesions                                     99
  TOTAL                                                  7007

Cible par classe (mediane des comptes originaux) : 1077
Facteur max de sur-echantillonnage : x5

Distribution du trainset apres equilibrage :
  [0] actinic keratoses and intraepithelial carcinoma    1077
  [1] basal ce